In [1]:
import pandas as pd
import numpy as np
from typing import List, Tuple, Dict, Optional
import logging

logging.basicConfig(level=logging.INFO, format='%(asctime)s - %(levelname)s - %(message)s')
logger = logging.getLogger(__name__)

- X: 特徵矩陣
- y_pct: pct_of_cap 目標變數
- y_yrs: YRS 目標變數
- ids: 識別欄位
- group_a: 純實力特徵
- group_b: 市場雜訊特徵

In [5]:
df = pd.read_csv('../../data/processed/featured_nba_data.csv')

# 在這裡才把資料分開
ids = df[['Player', 'year']]
y_pct = df['Cap_Pct']
y_yrs = df['YRS']

# 定義要丟給模型的特徵
features_to_drop = ['Player', 'year', 'Cap_Pct', 'YRS']
X = df.drop(columns=features_to_drop)

# 4. 依照「時間」切分訓練集與測試集 (Time-based Split)
# 預設用 <= 2022 當訓練集，> 2022 當測試集
train_mask = ids['year'] <= 2022
test_mask = ids['year'] > 2022

X_train, X_test = X[train_mask], X[test_mask]
y_pct_train, y_pct_test = y_pct[train_mask], y_pct[test_mask]
y_yrs_train, y_yrs_test = y_yrs[train_mask], y_yrs[test_mask]

# 測試集的 ids 非常重要，SHAP 畫圖會用到
ids_test = ids[test_mask]

## model_building

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import shap
import pickle
import os
import logging

from sklearn.model_selection import train_test_split
from xgboost import XGBRegressor
from sklearn.experimental import enable_halving_search_cv  
from sklearn.model_selection import HalvingGridSearchCV
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_squared_error, r2_score, accuracy_score

# Set up logging
logging.basicConfig(level=logging.INFO, format='%(asctime)s - %(levelname)s - %(message)s')
logger = logging.getLogger(__name__)

# ==========================================
# 1. 訓練與解釋函式
# ==========================================
def train_pricing_model(X_train: pd.DataFrame, y_train: pd.Series, X_test: pd.DataFrame, y_test: pd.Series):
    logger.info("開始 HalvingGridSearchCV 調參 XGBoost...")
    reg = XGBRegressor(learning_rate=0.1, random_state=42) 
    
    param_grid = {
        "max_depth": [3, 5, 7, 9],
        "min_child_weight": [1, 5, 10],
        "subsample": [0.8, 1.0],
        "colsample_bytree": [0.8, 1.0]
    }
    
    search = HalvingGridSearchCV(
        estimator=reg, param_grid=param_grid, resource='n_estimators', 
        min_resources=50, max_resources=800, factor=2, cv=3, 
        scoring='neg_root_mean_squared_error', random_state=42, n_jobs=-1
    )
    search.fit(X_train, y_train)
    
    logger.info(f"🏆 最佳參數組合: {search.best_params_}")
    best_model = search.best_estimator_
    
    y_pred = best_model.predict(X_test)
    r2 = r2_score(y_test, y_pred)
    rmse = np.sqrt(mean_squared_error(y_test, y_pred))
    
    logger.info(f"定價模型測試集表現 -> R2: {r2:.4f}, RMSE: {rmse:.4f}")
    return best_model

def train_duration_model(X_train: pd.DataFrame, y_train: pd.Series, X_test: pd.DataFrame, y_test: pd.Series):
    logger.info("開始訓練年限模型 (已加入 StandardScaler)...")
    pipeline = Pipeline([
        ('scaler', StandardScaler()),
        ('classifier', LogisticRegression(multi_class='multinomial', solver='lbfgs', max_iter=1000, random_state=42))
    ])
    pipeline.fit(X_train, y_train)
    y_pred = pipeline.predict(X_test)
    acc = accuracy_score(y_test, y_pred)
    logger.info(f"年限模型測試集表現 -> Accuracy: {acc:.4f}")
    return pipeline

def extract_pure_skill(model, X_test: pd.DataFrame, ids_test: pd.DataFrame, group_a: list, group_b: list):
    logger.info("開始執行 SHAP 雜訊剝離...")
    explainer = shap.TreeExplainer(model)
    shap_values_obj = explainer(X_test) 
    base_value = explainer.expected_value
    results = []
    
    for i in range(len(X_test)):
        player_name = ids_test.iloc[i]['Player']
        contract_year = ids_test.iloc[i]['year']
        player_shap_vals = shap_values_obj.values[i]
        
        skill_impact = sum([player_shap_vals[X_test.columns.get_loc(f)] for f in group_a if f in X_test.columns])
        pure_skill_pct = base_value + skill_impact
        noise_impact = sum([player_shap_vals[X_test.columns.get_loc(f)] for f in group_b if f in X_test.columns])
        
        results.append({
            'Player': player_name, 'Year': contract_year, 'Base_Value': base_value,
            'Pure_Skill_Impact': skill_impact, 'Pure_Skill_Valuation': pure_skill_pct,
            'Market_Noise_Impact': noise_impact, 'Total_Predicted_Cap_Pct': pure_skill_pct + noise_impact
        })
    return explainer, shap_values_obj, pd.DataFrame(results)

def plot_player_shap_waterfall(shap_values_obj, X_test: pd.DataFrame, ids_test: pd.DataFrame, player_name: str, target_year: int, output_dir: str = '../../reports/figures'):
    match_idx = np.where((ids_test['Player'] == player_name) & (ids_test['year'] == target_year))[0]
    if len(match_idx) == 0:
        logger.warning(f"圖表生成失敗：找不到 {player_name} 在 {target_year} 年的資料。")
        return
    
    plt.figure(figsize=(10, 6))
    shap.plots.waterfall(shap_values_obj[match_idx[0]], max_display=10, show=False)
    plt.title(f"Contract Valuation Breakdown: {player_name.title()} ({target_year})", fontsize=14, fontweight='bold', pad=20)
    plt.tight_layout()
    
    if not os.path.exists(output_dir): os.makedirs(output_dir)
    file_path = os.path.join(output_dir, f"shap_waterfall_{player_name.replace(' ', '_').lower()}_{target_year}.png")
    plt.savefig(file_path, dpi=300, bbox_inches='tight')
    logger.info(f"已儲存 SHAP 瀑布圖至: {file_path}")
    plt.close()

# ==========================================
# 2. 主程式執行管線
# ==========================================
if __name__ == "__main__":
    # --- A. 讀取資料 ---
    data_path = r'C:\Users\user\Python\Final_test\data\processed\featured_nba_data.csv'
    logger.info(f"讀取特徵資料庫: {data_path}")
    df = pd.read_csv(data_path)
    
    # --- B. 類別特徵 One-Hot Encoding ---
    logger.info("執行類別特徵 One-Hot Encoding...")
    df = pd.get_dummies(df, columns=['Team', 'Pos'], drop_first=False)
    bool_cols = df.select_dtypes(include=['bool']).columns
    df[bool_cols] = df[bool_cols].astype(int)
    
    # --- C. 分離特徵與目標 ---
    ids = df[['Player', 'year']]
    y_pct = df['Cap_Pct']
    y_yrs = df['YRS']
    
    # 🚨 [降級修改]：找出所有 health 相關特徵並將其剔除
    health_cols = [c for c in df.columns if c.endswith('_health')]
    logger.info(f"🔧 暫時降級模型，移除以下健康特徵: {health_cols}")
    
    # 將不要的標籤與健康特徵合併在一起 drop 掉
    cols_to_drop = ['Player', 'year', 'Cap_Pct', 'YRS'] + health_cols
    X = df.drop(columns=cols_to_drop, errors='ignore')
    
    # --- D. 定義 Group A (實力) 與 Group B (市場雜訊) ---
    team_cols = [c for c in X.columns if c.startswith('Team_')]
    group_b = ['is_retained', 'Payroll_Pct', 'Cap_Space_Pct'] + team_cols
    # 因為前面 X 已經排除了 health，所以 group_a 也不會抓到 health 特徵了
    group_a = [c for c in X.columns if c not in group_b]
    
    # --- E. 切割訓練集與測試集 ---
    X_train, X_test, y_pct_train, y_pct_test, y_yrs_train, y_yrs_test, ids_train, ids_test = train_test_split(
        X, y_pct, y_yrs, ids, test_size=0.15, random_state=42
    )
    
    # --- F. 開始訓練 ---
    best_pricing_model = train_pricing_model(X_train, y_pct_train, X_test, y_pct_test)
    pipeline_duration_model = train_duration_model(X_train, y_yrs_train, X_test, y_yrs_test)
    
    # --- G. 匯出訓練好的大腦 ---
    models_dir = r'C:\Users\user\Python\Final_test\src\models'
    if not os.path.exists(models_dir): os.makedirs(models_dir)
        
    with open(os.path.join(models_dir, 'pricing_model.pkl'), 'wb') as f:
        pickle.dump(best_pricing_model, f)
    with open(os.path.join(models_dir, 'duration_model.pkl'), 'wb') as f:
        pickle.dump(pipeline_duration_model, f)
        
    logger.info("✅ 定價與年限模型已存檔完畢！")

    # --- H. SHAP 解析與圖表測試 ---
    explainer, shap_values_obj, shap_results_df = extract_pure_skill(best_pricing_model, X_test, ids_test, group_a, group_b)
    
    with open(os.path.join(models_dir, 'shap_explainer.pkl'), 'wb') as f:
        pickle.dump(explainer, f)
    logger.info("✅ SHAP 解釋器已存檔完畢！")
    
    # 隨機挑一位測試集裡的知名球員來畫圖
    test_player = ids_test.iloc[0]['Player']
    test_year = ids_test.iloc[0]['year']
    plot_player_shap_waterfall(shap_values_obj, X_test, ids_test, player_name=test_player, target_year=test_year)

2026-06-05 01:05:22,868 - INFO - 讀取特徵資料庫: C:\Users\user\Python\Final_test\data\processed\featured_nba_data.csv
2026-06-05 01:05:22,882 - INFO - 執行類別特徵 One-Hot Encoding...
2026-06-05 01:05:22,890 - INFO - 🔧 暫時降級模型，移除以下健康特徵: ['ATTENDANCE_RATE_health', 'MAJOR_INJURY_health', 'MIN_PER_GAME_health']
2026-06-05 01:05:22,894 - INFO - 開始 HalvingGridSearchCV 調參 XGBoost...
2026-06-05 01:06:04,354 - INFO - 🏆 最佳參數組合: {'colsample_bytree': 1.0, 'max_depth': 5, 'min_child_weight': 1, 'subsample': 0.8, 'n_estimators': 800}
2026-06-05 01:06:04,374 - INFO - 定價模型測試集表現 -> R2: 0.7660, RMSE: 0.0410
2026-06-05 01:06:04,375 - INFO - 開始訓練年限模型 (已加入 StandardScaler)...
c:\Users\user\anaconda3\Lib\site-packages\sklearn\linear_model\_logistic.py:1272: FutureWarning: 'multi_class' was deprecated in version 1.5 and will be removed in 1.8. From then on, it will always use 'multinomial'. Leave it to its default value to avoid this warning.
  warnings.warn(
2026-06-05 01:06:04,507 - INFO - 年限模型測試集表現 -> Accuracy: 0.4963
